In [1]:
import cv2
import numpy as np

CAMERA_INDEX = 0
MAX_FEATURES = 500   # fewer = faster, still plenty for flow

sift = cv2.SIFT_create(nfeatures=MAX_FEATURES)

lk_params = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)
)

REDETECT_EVERY = 15   # re-run SIFT every N frames to refresh keypoints

def get_keypoints(gray):
    kps = sift.detect(gray, None)   # only detect, no descriptor needed for LK
    pts = np.array([kp.pt for kp in kps], dtype=np.float32).reshape(-1, 1, 2)
    return pts

def draw_flow(frame, pts1, pts2, status):
    vis = frame.copy()
    good1 = pts1[status == 1]
    good2 = pts2[status == 1]
    for (x0, y0), (x1, y1) in zip(good1.reshape(-1, 2).astype(int),
                                   good2.reshape(-1, 2).astype(int)):
        cv2.arrowedLine(vis, (x0, y0), (x1, y1), (0, 255, 0), 2, tipLength=0.4)
        cv2.circle(vis, (x0, y0), 3, (0, 0, 255), -1)
    tracked = int(status.sum())
    cv2.putText(vis, f"Tracking {tracked} points", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)
    return vis

def run():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print("Could not open camera.")
        return

    prev_gray = None
    prev_pts  = None
    frame_idx = 0

    print("Press Q to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Re-detect SIFT keypoints every N frames (or on first frame)
        if frame_idx % REDETECT_EVERY == 0 or prev_pts is None:
            prev_pts = get_keypoints(gray)
            print(f"[{frame_idx:05d}]  Detected {len(prev_pts)} keypoints")

        if prev_gray is not None and prev_pts is not None and len(prev_pts) > 0:
            # Track points from prev frame to current using Lucas-Kanade
            curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(
                prev_gray, gray, prev_pts, None, **lk_params
            )
            status = status.flatten()

            flow_vis = draw_flow(frame, prev_pts, curr_pts, status)
            cv2.imshow("Optical Flow (SIFT + LK)", flow_vis)

            # Keep only successfully tracked points for next iteration
            prev_pts = curr_pts[status == 1].reshape(-1, 1, 2)

        prev_gray = gray
        frame_idx += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run()

Press Q to quit.
[00000]  Detected 500 keypoints
[00015]  Detected 462 keypoints
[00030]  Detected 385 keypoints
[00045]  Detected 336 keypoints
[00060]  Detected 306 keypoints
[00075]  Detected 374 keypoints
[00090]  Detected 264 keypoints
[00105]  Detected 122 keypoints
[00120]  Detected 114 keypoints
[00135]  Detected 248 keypoints
[00150]  Detected 189 keypoints
[00165]  Detected 286 keypoints
[00180]  Detected 314 keypoints
[00195]  Detected 108 keypoints
[00210]  Detected 98 keypoints
[00225]  Detected 262 keypoints
[00240]  Detected 188 keypoints
[00255]  Detected 391 keypoints
[00270]  Detected 311 keypoints
[00285]  Detected 461 keypoints
[00300]  Detected 262 keypoints
[00315]  Detected 301 keypoints
[00330]  Detected 500 keypoints
[00345]  Detected 474 keypoints
[00360]  Detected 347 keypoints
[00375]  Detected 390 keypoints
[00390]  Detected 403 keypoints
[00405]  Detected 416 keypoints
[00420]  Detected 364 keypoints
[00435]  Detected 367 keypoints
